# Journal Overlap & APC Exposure Analysis
## NLM MEDLINE Journals · PMC Journals · NIH-Funded Journals · Northwestern TAs · Northwestern Publications

**Purpose:** Compare five journal lists using ISSN as the primary match key. Classify each journal's open-access and APC cost profile. Estimate Northwestern's annual and per-year APC exposure for NIH-funded publications.

**Intended audience:** Library leadership, research administration, scholarly communications policy.

**Analyst:** Galter Health Sciences Library — Metrics and Impact Core  
**Last run:** See Configuration cell output.


## 1. Terminology Guide

**Read this section before interpreting any outputs.** Several terms in this domain are easy to conflate. This notebook uses the following precise definitions throughout.

---

### PMC (PubMed Central) — the repository
PMC is NCBI's full-text archive of biomedical literature. Under the NIH Public Access Policy, researchers funded by NIH are *required to deposit accepted manuscripts in PMC*. This is an article-level obligation and applies regardless of which journal the paper is published in.

**This analysis does NOT use PMC as an article-level deposit tracker.** 

---

### PMC Journals — the journal participation list
A subset of journals have formal agreements with NLM to deposit content directly into PMC. These journals appear on the **PMC Journal List** (published by NCBI). Being on this list means the *journal* has an agreement with PMC — not that individual articles have been deposited.

**In this analysis, "PMC Journals" refers to journals on the PMC Journal List.** This is important for APC liability classification: if a hybrid journal has a PMC agreement with immediate release, authors may be able to satisfy NIH public access requirements through the journal deposit rather than paying for hybrid OA.

> **Source:** `https://cdn.ncbi.nlm.nih.gov/pmc/home/jlist.csv`  
> The flag `in_pmc_journals` = True means the journal appears on this list.

---

### NLM Catalog — the full journal registry
The NLM Catalog contains bibliographic records for over 1.5 million items including journals, books, and databases. It includes historical, ceased, and non-English journals across all of medicine and related fields.

---

### NLM MEDLINE Journals — the subset used here
This analysis uses the query `currentlyindexed[All]` against the NLM Catalog, which returns only journals **currently indexed in MEDLINE** (~5,200 journals). MEDLINE is the selective database of peer-reviewed biomedical literature; it is a curated subset of the NLM Catalog.

**In this analysis, "NLM MEDLINE Journals" refers to this MEDLINE-indexed subset.** It is NOT the full NLM Catalog. Journals not indexed in MEDLINE are not present in the NLM MEDLINE Journals list and will appear only in other source lists.

> **Source:** NCBI E-utilities, query `currentlyindexed[All]`  
> The flag `in_nlm_medline` = True means the journal is currently indexed in MEDLINE.

---

### Summary table of source lists used in this analysis

| Variable label | What it represents | Source |
|---|---|---|
| **NLM MEDLINE Journals** | Journals currently indexed in MEDLINE | NCBI E-utilities |
| **PMC Journals** | Journals with formal PMC deposit agreements | NCBI CDN jlist.csv |
| **NIH-Funded Journals** | Top 2,228 journals by NIH paper volume | Dimensions, Jan–Jul 2025 |
| **Northwestern TA Journals** | Journals in NU Wiley or SN BTAA agreements | Publisher-supplied files |
| **Northwestern Publications** | NU NIH-funded papers from NIH Reporter | NIH Reporter, FY2020–present |


## 2. Assumptions and Known Limitations

### Matching
- **ISSN is the sole match key.** Title-based fallback is not performed. Journals with inconsistent ISSNs across sources will be missed.
- **TA files contain eISSN only.** Journals with print-only ISSNs may be falsely classified as not TA-covered. This means TA coverage counts are conservative.
- **ISSN instability.** ISSNs can change with title changes. NLM's linking ISSN partially mitigates this.

### Source scope
- **NLM MEDLINE Journals** covers only currently MEDLINE-indexed journals. Non-MEDLINE biomedical journals (including many society and open-access journals) are absent.
- **NIH-Funded Journals** covers Jan–Jul 2025 only — 7 months, not a full year. Journals with H2-heavy publication patterns are underrepresented.
- **NIH-Funded Journals** covers only the top 2,228 journals. NU publications in journals outside this list have no APC pricing data. The coverage gap is quantified in the Validation section.
- **NIH Reporter completeness.** Known reporting lags; affiliation matching is imperfect.

### Financial estimates
- **All APC estimates use 2025 list prices regardless of publication year.** Historical prices are not available. Applying 2025 prices to 2020 publications overstates past exposure; applying them to future years is a planning assumption, not a forecast.
- **List prices only.** Institutional discounts, member pricing, and individual waivers are not reflected.
- **APC liability classification** assumes the author uses the lowest-cost OA route available. Authors who voluntarily pay hybrid OA APCs where PMC immediate release is available are not captured.
- **Journals not in the NIH 2,228 list have unknown APC prices** and contribute $0 to financial totals.

### What this analysis supports vs. does not support
| Supports | Does NOT support |
|---|---|
| Identifying where NU NIH-funded publishing is concentrated | Calculating actual historical APC expenditure |
| Flagging TA coverage and potential unwaived exposure | Proving TA agreements are being used |
| Estimating annual APC exposure (order of magnitude) | Precise budget forecasting |
| Year-by-year publication volume trends | Future publication volume prediction |


## 3. Imports

In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import time
import pickle
import re
import io
from pathlib import Path
from datetime import datetime

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ── NCBI credentials ──────────────────────────────────────────────────────────
from config import ENTREZ_EMAIL, ENTREZ_API_KEY

print("Imports OK")
print(f"Run date/time: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


Imports OK
Run date/time: 2026-04-01 17:33


## 4. Configuration

In [3]:

# ── NLM MEDLINE Journals query ────────────────────────────────────────────────
# "currentlyindexed[All]" = journals currently indexed in MEDLINE (~5,200)
# "ncbijournals[All]"     = broader NLM Catalog journal collection (~60,000)
# NOTE: Changing this requires deleting cache/nlm_medline_journals.pkl.
NLM_QUERY = "currentlyindexed[All]"

# ── External file paths ───────────────────────────────────────────────────────
PMC_JOURNALS_URL = "https://cdn.ncbi.nlm.nih.gov/pmc/home/jlist.csv"
NIH_FILE         = Path("data/NIH2025_2228TopJournals.csv")
WILEY_FILE       = Path("data/2025_08-25 Wiley Hybrid and OA Journals.csv")
SN_FILE          = Path("data/2025_BTAA_Springer_Nature_hybrid_journals.csv")
NU_FILE          = Path("data/2025_09-05 Northwestern Pubs from NIH Reporter 2020 to present_All FY.csv")

# ── E-utilities ───────────────────────────────────────────────────────────────
EUTILS     = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
BATCH_SIZE = 500
SLEEP_SEC  = 0.11 if ENTREZ_API_KEY else 0.34

# ── Directories ───────────────────────────────────────────────────────────────
CACHE_DIR  = Path("cache")
OUTPUT_DIR = Path("output")
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

NLM_CACHE = CACHE_DIR / "nlm_medline_journals.pkl"
PMC_CACHE = CACHE_DIR / "pmc_journals.pkl"

# ── File existence check ──────────────────────────────────────────────────────
print("File existence check:")
for label, path in [("NIH top journals", NIH_FILE), ("Wiley TA", WILEY_FILE),
                    ("Springer Nature TA", SN_FILE), ("NU publications", NU_FILE)]:
    status = "OK" if path.exists() else "MISSING ⚠"
    print(f"  [{status}] {label}: {path}")

print(f"\nNLM query : {NLM_QUERY}")
print(f"API key   : {'set' if ENTREZ_API_KEY else 'not set — 3 req/sec limit applies'}")


File existence check:
  [OK] NIH top journals: data\NIH2025_2228TopJournals.csv
  [OK] Wiley TA: data\2025_08-25 Wiley Hybrid and OA Journals.csv
  [OK] Springer Nature TA: data\2025_BTAA_Springer_Nature_hybrid_journals.csv
  [OK] NU publications: data\2025_09-05 Northwestern Pubs from NIH Reporter 2020 to present_All FY.csv

NLM query : currentlyindexed[All]
API key   : set


## 5. Helper Functions

In [4]:
def normalize_issn(raw):
    """
    Normalize an ISSN string to XXXX-XXXX uppercase format.
    Returns None if the input cannot be parsed as a valid 8-character ISSN.
    """
    if not raw or not isinstance(raw, str):
        return None
    digits = re.sub(r'[^0-9Xx]', '', raw)
    if len(digits) == 8:
        return f"{digits[:4]}-{digits[4:]}".upper()
    return None


def any_issn_in_set(row, issn_cols, target_set):
    """
    Return True if any normalized ISSN from the specified columns of a row
    appears in target_set. Used for cross-source membership classification.
    """
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in target_set:
            return True
    return False


def read_csv_robust(filepath, sep=None, **kwargs):
    """
    Attempt to read a CSV file trying utf-8-sig, cp1252, latin-1 in sequence.
    Returns (DataFrame, encoding_used). Raises ValueError if all encodings fail.
    """
    for enc in ['utf-8-sig', 'cp1252', 'latin-1']:
        try:
            df = pd.read_csv(filepath, sep=sep, engine='python',
                             dtype=str, encoding=enc, **kwargs)
            print(f"  Loaded: {filepath.name} | {len(df):,} rows | encoding={enc}")
            return df, enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {filepath} with utf-8-sig, cp1252, or latin-1")


def entrez_params(extras=None):
    """Return base E-utilities parameter dict merged with any extras."""
    p = {"email": ENTREZ_EMAIL}
    if ENTREZ_API_KEY:
        p["api_key"] = ENTREZ_API_KEY
    if extras:
        p.update(extras)
    return p


# ── Unit tests for normalize_issn ────────────────────────────────────────────
_tests = [("2041-1723","2041-1723"),("20411723","2041-1723"),
          ("2041 1723","2041-1723"),("XXXX",None),(None,None)]
assert all(normalize_issn(r)==e for r,e in _tests), "normalize_issn unit test failed"
print("Helper functions OK — normalize_issn unit tests passed.")


Helper functions OK — normalize_issn unit tests passed.


## 6. Load: NIH-Funded Journals (Top 2,228)

**What this is:** The top 2,228 journals by volume of NIH-funded papers, January–July 2025.  
**Source:** Dimensions bibliometric data as of 2025-07-24.  
**Key limitation:** 7-month partial year only; H2-heavy journals are underrepresented.  
**APC data:** 2025 list prices from publisher-reported sources. List prices ≠ prices paid.


In [5]:
def load_nih_journals(filepath):
    """
    Load the NIH-funded journals file.
    Extracts up to 4 ISSN columns. Returns a cleaned DataFrame.
    """
    df, _ = read_csv_robust(filepath, sep=None)
    print(f"  Columns: {list(df.columns)}")

    orig_cols = list(df.columns)

    # Normalize column names to snake_case for mapping
    df.columns = (
        df.columns.str.strip().str.lower()
          .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    )

    rename_map = {
        next((c for c in df.columns if 'order' in c), None)                                      : 'nih_order',
        next((c for c in df.columns if c == 'journal'), None)                                    : 'nih_journal',
        next((c for c in df.columns if 'publisher' in c and 'type' not in c), None)              : 'nih_publisher',
        next((c for c in df.columns if 'publications' in c and 'nih' in c), None)                : 'nih_pub_count',
        next((c for c in df.columns if 'oa_status' in c or ('oa' in c and 'status' in c)), None) : 'nih_oa_status',
        next((c for c in df.columns if 'publisher_type' in c), None)                             : 'nih_publisher_type',
        next((c for c in df.columns if 'apc_2025' in c or ('apc' in c and '2025' in c)), None)  : 'nih_apc_2025_usd',
        next((c for c in df.columns if 'apc_category' in c), None)                              : 'nih_apc_category',
    }
    rename_map = {k: v for k, v in rename_map.items() if k is not None}
    df = df.rename(columns=rename_map)

    # Extract ISSN1–ISSN4 by matching original column names (case-insensitive)
    orig_lower = [c.strip().lower() for c in orig_cols]
    for i in range(1, 5):
        target = f'issn{i}'
        if target in orig_lower:
            orig_col = orig_cols[orig_lower.index(target)]
            snake    = re.sub(r'[^a-z0-9]+', '_', orig_col.strip().lower()).strip('_')
            df[f'nih_issn{i}'] = df.get(snake, pd.Series(dtype=str)).apply(normalize_issn)
        else:
            df[f'nih_issn{i}'] = None

    for col in ['nih_pub_count', 'nih_apc_2025_usd', 'nih_order']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df


nih_df = load_nih_journals(NIH_FILE)

# Validate ISSN extraction
issn_cols_nih = ['nih_issn1', 'nih_issn2', 'nih_issn3', 'nih_issn4']
rows_with_issn = nih_df[issn_cols_nih].notna().any(axis=1).sum()
print(f"  Rows with ≥1 valid ISSN: {rows_with_issn:,} / {len(nih_df):,}")
if rows_with_issn == 0:
    print("  ⚠ WARNING: No ISSNs extracted from NIH file — check column names above.")

# Build NIH ISSN lookup: ISSN → NIH metadata dict
NIH_META_COLS = [c for c in ['nih_order','nih_journal','nih_publisher','nih_pub_count',
                               'nih_oa_status','nih_publisher_type','nih_apc_2025_usd',
                               'nih_apc_category'] if c in nih_df.columns]
NIH_EMPTY = {c: None for c in NIH_META_COLS}

nih_lookup     = {}
nih_dup_issns  = []
for _, row in nih_df.iterrows():
    entry = {c: row.get(c) for c in NIH_META_COLS}
    for col in issn_cols_nih:
        v = row.get(col)
        if pd.notna(v) and v:
            if v in nih_lookup:
                nih_dup_issns.append(v)
            nih_lookup[v] = entry

nih_issns = set(nih_lookup.keys())
print(f"\nNIH lookup: {len(nih_issns):,} unique ISSNs")
if nih_dup_issns:
    print(f"  Note: {len(set(nih_dup_issns))} ISSNs appeared in >1 NIH row (last-write-wins)")


  Loaded: NIH2025_2228TopJournals.csv | 2,228 rows | encoding=cp1252
  Columns: ['Order (# NIH papers 01-07/2025)', 'Journal', 'ISSN1', 'ISSN2', 'ISSN3', 'ISSN4', 'Publisher', 'Publications acknowledging NIH funding (01-07/2025)', 'Journal OA status', 'Publisher type', 'APC 2025 (USD)', 'APC category', 'Total APCs (based on 01-07/2025)', 'APCs covered by $2k cap', '% covered by $2k cap', 'APCs not covered by $2k cap', 'APCs covered by $3k cap', '% covered by $3k cap', 'APCs not covered by $3k cap', 'APCs covered by $6 cap', '% covered by $6 cap', 'APCs not covered by $6 cap']
  Rows with ≥1 valid ISSN: 2,228 / 2,228

NIH lookup: 3,860 unique ISSNs
  Note: 27 ISSNs appeared in >1 NIH row (last-write-wins)


## 7. Load: Northwestern NIH-Funded Publications

**What this is:** NU publication records linked to NIH-funded grants, exported from NIH Reporter.  
**Unit:** One row per publication. Aggregated to journal level (by ISSN) and also to journal×year level.  
**Period:** Fiscal years 2020–present (exact range printed at runtime).  
**Known limitation:** NIH Reporter has reporting lags; recent years may be undercounted.

### Two aggregations produced:
- **`nu_agg`** — journal-level totals (all years combined): `nu_pub_count`, `nu_unique_grants`
- **`nu_yearly_agg`** — journal × year: `nu_pub_count_year` (used for per-year APC trend analysis)


In [6]:
def load_nu_journals(filepath):
    """
    Load NU publication data.
    Returns: (journal_agg_df, yearly_agg_df, n_years, year_min, year_max)
      - journal_agg_df : one row per ISSN, all-years totals
      - yearly_agg_df  : one row per (ISSN, pub_year)
    """
    df, _ = read_csv_robust(filepath, sep=None)
    print(f"  Columns: {list(df.columns)}")

    issn_col  = next((c for c in df.columns if c.strip().upper() == 'ISSN'), None)
    grant_col = next((c for c in df.columns if 'core project' in c.lower()), None)
    year_col  = next((c for c in df.columns if 'pub year' in c.lower()), None)

    if not issn_col:
        raise ValueError(f"ISSN column not found in {filepath}. Columns: {list(df.columns)}")

    print(f"  ISSN col  : {issn_col}")
    print(f"  Grant col : {grant_col}")
    print(f"  Year col  : {year_col}")

    df['issn_norm'] = df[issn_col].apply(normalize_issn)

    before = len(df)
    df = df[df['issn_norm'].notna()].copy()
    print(f"  Rows with valid ISSN: {len(df):,} (dropped {before-len(df):,})")

    # ── Year range ────────────────────────────────────────────────────────────
    if year_col:
        df['pub_year_num'] = pd.to_numeric(df[year_col], errors='coerce')
        years = df['pub_year_num'].dropna().unique()
        if len(years) > 0:
            n_years   = int(len(years))
            year_min  = int(years.min())
            year_max  = int(years.max())
        else:
            print("  ⚠ Year column present but no valid values found. n_years defaulting to 1.")
            n_years, year_min, year_max = 1, None, None
    else:
        print("  ⚠ Pub Year column not found. Per-year breakdown unavailable. n_years=1.")
        df['pub_year_num'] = np.nan
        n_years, year_min, year_max = 1, None, None

    print(f"  Publication years: {year_min}–{year_max} ({n_years} year(s))")

    # ── Journal-level aggregate (all years) ───────────────────────────────────
    agg_dict = {'nu_pub_count': ('issn_norm', 'count')}
    if grant_col:
        agg_dict['nu_unique_grants'] = (grant_col, 'nunique')
    journal_agg = df.groupby('issn_norm').agg(**agg_dict).reset_index()
    print(f"  Unique journals (by ISSN, all years): {len(journal_agg):,}")

    # ── Journal × year aggregate ──────────────────────────────────────────────
    if year_col and year_min is not None:
        yr_agg_dict = {'nu_pub_count_year': ('issn_norm', 'count')}
        yearly_agg = (
            df[df['pub_year_num'].notna()]
            .groupby(['issn_norm', 'pub_year_num'])
            .agg(**yr_agg_dict)
            .reset_index()
            .rename(columns={'pub_year_num': 'pub_year'})
        )
        yearly_agg['pub_year'] = yearly_agg['pub_year'].astype(int)
        print(f"  Journal×year rows: {len(yearly_agg):,}")
    else:
        yearly_agg = pd.DataFrame(columns=['issn_norm','pub_year','nu_pub_count_year'])
        print("  Per-year breakdown unavailable (no year column).")

    return journal_agg, yearly_agg, n_years, year_min, year_max


nu_agg, nu_yearly_agg, nu_years, nu_year_min, nu_year_max = load_nu_journals(NU_FILE)

# ── Build NU ISSN lookup (journal level) ──────────────────────────────────────
NU_META_COLS = [c for c in ['nu_pub_count','nu_unique_grants'] if c in nu_agg.columns]
NU_EMPTY     = {c: None for c in NU_META_COLS}
nu_lookup    = {row['issn_norm']: {c: row.get(c) for c in NU_META_COLS}
                for _, row in nu_agg.iterrows()}
nu_issns     = set(nu_lookup.keys())

print(f"\nNU lookup: {len(nu_issns):,} unique journals")
print(f"Total NU-NIH publications (all years): {nu_agg['nu_pub_count'].sum():,.0f}")


  Loaded: 2025_09-05 Northwestern Pubs from NIH Reporter 2020 to present_All FY.csv | 27,474 rows | encoding=utf-8-sig
  Columns: ['Core Project Number', 'Affiliation', 'Pub Year', 'Authors', 'Country', 'ISSN', 'Journal Issue', 'Journal (Link to PubMed abstract)', 'Journal Title ABBR', 'Journal Volume', 'Language', 'Page Number', 'PMC ID', 'PMID', 'PUB Date', 'Title (Link to full-text in PubMed Central)', 'Related Publications in PubMed', 'Related Publications in Google Scholar', 'Articles Citing from PubMed Central', 'Articles Citing from Google Scholar', 'Relative Citation Ratio']
  ISSN col  : ISSN
  Grant col : Core Project Number
  Year col  : Pub Year
  Rows with valid ISSN: 26,913 (dropped 561)
  Publication years: 2020–2025 (6 year(s))
  Unique journals (by ISSN, all years): 2,522
  Journal×year rows: 6,041

NU lookup: 2,522 unique journals
Total NU-NIH publications (all years): 26,913


## 8. Load: Northwestern Transformative Agreement Journal Lists

**What these are:** Lists of journals covered by Northwestern's transformative agreements with Wiley and Springer Nature (BTAA). APCs are waived for NU corresponding authors in covered journals.  
**Key limitation:** Both files contain eISSN only; print-only ISSN journals may be missed.


In [7]:
def load_wiley_ta(filepath):
    df, _ = read_csv_robust(filepath, sep=None)
    print(f"  Columns: {list(df.columns)}")
    df.columns = (df.columns.str.strip().str.lower()
                    .str.replace(r'[^a-z0-9]+','_',regex=True).str.strip('_'))
    rename_map = {}
    for c in df.columns:
        if re.search(r'title', c):          rename_map[c] = 'ta_title'
        elif re.search(r'issn', c):         rename_map[c] = 'ta_issn_raw'
        elif re.search(r'type|model', c):   rename_map[c] = 'ta_publishing_model'
    df = df.rename(columns=rename_map)
    if 'ta_issn_raw' not in df.columns:
        raise ValueError(f"ISSN column not found in Wiley file. Columns: {list(df.columns)}")
    df['ta_issn']      = df['ta_issn_raw'].apply(normalize_issn)
    df['ta_agreement'] = 'Wiley'
    before = len(df)
    df = df[df['ta_issn'].notna()].copy()
    print(f"  Rows with valid ISSN: {len(df):,} (dropped {before-len(df):,})")
    return df[['ta_title','ta_issn','ta_publishing_model','ta_agreement']].copy()


def load_sn_ta(filepath):
    df, _ = read_csv_robust(filepath, sep=None)
    print(f"  Columns: {list(df.columns)}")
    df.columns = (df.columns.str.strip().str.lower()
                    .str.replace(r'[^a-z0-9]+','_',regex=True).str.strip('_'))
    rename_map = {}
    for c in df.columns:
        if re.search(r'title',c) and 'imprint' not in c: rename_map[c] = 'ta_title'
        elif re.search(r'^e_?issn$|eissn',c):            rename_map[c] = 'ta_issn_raw'
        elif re.search(r'publishing_model|model',c):      rename_map[c] = 'ta_publishing_model'
        elif re.search(r'oa_license|license',c):          rename_map[c] = 'ta_oa_license'
    df = df.rename(columns=rename_map)
    if 'ta_issn_raw' not in df.columns:
        raise ValueError(f"eISSN column not found in SN file. Columns: {list(df.columns)}")
    df['ta_issn']      = df['ta_issn_raw'].apply(normalize_issn)
    df['ta_agreement'] = 'Springer Nature'
    before = len(df)
    df = df[df['ta_issn'].notna()].copy()
    print(f"  Rows with valid ISSN: {len(df):,} (dropped {before-len(df):,})")
    keep = ['ta_title','ta_issn','ta_publishing_model','ta_agreement']
    if 'ta_oa_license' in df.columns: keep.append('ta_oa_license')
    return df[keep].copy()


print("Loading Wiley TA:")
wiley_df = load_wiley_ta(WILEY_FILE)
print("\nLoading Springer Nature TA:")
sn_df    = load_sn_ta(SN_FILE)

ta_df = pd.concat([wiley_df, sn_df], ignore_index=True)

# Build TA lookup: ISSN → {northwestern_ta_agreement, ta_publishing_model, ta_oa_license}
TA_META_COLS = ['northwestern_ta_agreement','ta_publishing_model','ta_oa_license']
ta_lookup = {}
for _, row in ta_df.iterrows():
    issn  = row['ta_issn']
    if not issn: continue
    entry = {
        'northwestern_ta_agreement': row['ta_agreement'],
        'ta_publishing_model'       : row.get('ta_publishing_model', ''),
        'ta_oa_license'             : row.get('ta_oa_license', ''),
    }
    if issn in ta_lookup:
        if row['ta_agreement'] not in ta_lookup[issn]['northwestern_ta_agreement']:
            ta_lookup[issn]['northwestern_ta_agreement'] += f"; {row['ta_agreement']}"
    else:
        ta_lookup[issn] = entry

ta_issns = set(ta_lookup.keys())
TA_EMPTY = {c: None for c in TA_META_COLS}

overlap = sum(1 for v in ta_lookup.values() if ';' in v['northwestern_ta_agreement'])
print(f"\nWiley TA journals          : {len(wiley_df):,}")
print(f"Springer Nature TA journals: {len(sn_df):,}")
print(f"Combined unique TA ISSNs   : {len(ta_issns):,}")
if overlap: print(f"  Journals in BOTH agreements: {overlap:,}")


Loading Wiley TA:
  Loaded: 2025_08-25 Wiley Hybrid and OA Journals.csv | 1,847 rows | encoding=utf-8-sig
  Columns: ['Journal Title', 'Online ISSN', 'Type']
  Rows with valid ISSN: 1,847 (dropped 0)

Loading Springer Nature TA:
  Loaded: 2025_BTAA_Springer_Nature_hybrid_journals.csv | 2,041 rows | encoding=cp1252
  Columns: ['S/N', 'Journal ID', 'Journal Title', 'eISSN', 'Journal Imprint', 'Main Discipline', 'Publishing Model', 'OA License', 'URL']
  Rows with valid ISSN: 2,041 (dropped 0)

Wiley TA journals          : 1,847
Springer Nature TA journals: 2,041
Combined unique TA ISSNs   : 3,888


## 9. Load: PMC Journals

**What this is:** The list of journals with formal deposit agreements with PMC (the journal participation list).  
**Important distinction:** This is NOT the list of all articles deposited in PMC. It is the list of *journals* that have formal agreements with NLM to deposit content. Being on this list is relevant for APC classification because some journals' agreements include immediate-release deposit, which can satisfy NIH public access requirements without a paid OA APC.  
**Source:** NCBI CDN (`https://cdn.ncbi.nlm.nih.gov/pmc/home/jlist.csv`). Cached locally.


In [8]:
def fetch_pmc_journals(use_cache=True):
    """
    Download or load from cache the PMC Journals list.
    Stores fetch timestamp for staleness detection.
    """
    if use_cache and PMC_CACHE.exists():
        with open(PMC_CACHE,'rb') as f:
            cached = pickle.load(f)
        if isinstance(cached, dict):
            df       = cached['data']
            fetch_ts = cached.get('fetched','unknown')
            age_days = (pd.Timestamp.now()-pd.Timestamp(fetch_ts)).days if fetch_ts!='unknown' else None
            print(f"PMC Journals: loaded from cache (fetched {fetch_ts}, {age_days} days ago)")
            if age_days and age_days > 90:
                print(f"  ⚠ Cache is {age_days} days old — consider refresh with use_cache=False")
        else:
            df = cached
            print("PMC Journals: loaded from legacy cache (no timestamp) — consider refresh")
        return df

    print(f"Downloading PMC Journals list from {PMC_JOURNALS_URL} ...")
    resp = requests.get(PMC_JOURNALS_URL, timeout=60)
    resp.raise_for_status()
    try:
        df = pd.read_csv(io.StringIO(resp.content.decode('utf-8-sig')))
    except UnicodeDecodeError:
        df = pd.read_csv(io.StringIO(resp.content.decode('latin-1')))

    print(f"  Raw shape: {df.shape} | columns: {list(df.columns)}")

    df.columns = (df.columns.str.strip().str.lower()
                    .str.replace(r'[^a-z0-9]+','_',regex=True).str.strip('_'))

    col_map = {}
    for c in df.columns:
        if re.search(r'^journal_title$|^journal_name$|journal.*name',c):           col_map[c]='pmc_title'
        elif re.search(r'^e_?issn$|electronic.*issn|issn.*online|online.*issn',c): col_map[c]='pmc_eissn'
        elif re.search(r'^issn$|print.*issn|p_?issn|issn.*print',c):               col_map[c]='pmc_issn'
        elif re.search(r'nlm.*unique|nlm.*id',c):                                   col_map[c]='pmc_nlm_id'
        elif re.search(r'particip|agreement_status',c):                             col_map[c]='pmc_participation'
        elif re.search(r'embargo|release.*delay|delay.*release',c):                 col_map[c]='pmc_embargo'
    df = df.rename(columns=col_map)

    for col in ['pmc_title','pmc_issn','pmc_eissn']:
        if col not in df.columns: df[col] = np.nan

    df['pmc_issn_norm']  = df['pmc_issn'].apply(normalize_issn)
    df['pmc_eissn_norm'] = df['pmc_eissn'].apply(normalize_issn)

    before = len(df)
    df = df[df['pmc_issn_norm'].notna() | df['pmc_eissn_norm'].notna()].copy().reset_index(drop=True)
    print(f"  Rows with ISSN: {len(df):,} (dropped {before-len(df):,} no-ISSN rows)")

    with open(PMC_CACHE,'wb') as f:
        pickle.dump({'data':df,'fetched':pd.Timestamp.now().isoformat()},f)
    return df


pmc_df = fetch_pmc_journals(use_cache=True)
print(f"PMC Journals loaded: {len(pmc_df):,}")


PMC Journals: loaded from legacy cache (no timestamp) — consider refresh
PMC Journals loaded: 4,372


## 10. Load: NLM MEDLINE Journals

**What this is:** Journals currently indexed in MEDLINE, retrieved via NCBI E-utilities.  
**Important distinction:** This is NOT the full NLM Catalog. It is the subset of journals currently selected for MEDLINE indexing. MEDLINE is a curated database; the NLM Catalog contains far more records (~1.5M). The query `currentlyindexed[All]` returns approximately 5,200 journals.  
**Cache:** Stored locally with timestamp. Cache age printed at runtime.


In [9]:
def esearch_nlm(query):
    params = entrez_params({"db":"nlmcatalog","term":query,"usehistory":"y","retmax":0})
    resp = requests.get(f"{EUTILS}/esearch.fcgi",params=params,timeout=60)
    resp.raise_for_status()
    root = ET.fromstring(resp.text)
    return root.findtext('WebEnv'), root.findtext('QueryKey'), int(root.findtext('Count','0'))


def efetch_batch(webenv,query_key,start,retmax):
    params = entrez_params({"db":"nlmcatalog","query_key":query_key,"WebEnv":webenv,
                             "retstart":start,"retmax":retmax,"rettype":"xml","retmode":"xml"})
    resp = requests.get(f"{EUTILS}/efetch.fcgi",params=params,timeout=120)
    resp.raise_for_status()
    return resp.text


def parse_nlm_xml(xml_text):
    records = []
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError:
        print("  ⚠ XML parse error in batch — skipping")
        return records
    for rec in root.findall('.//NLMCatalogRecord'):
        nlm_id     = rec.findtext('NlmUniqueID','')
        title_el   = rec.find('.//TitleMain/Title')
        title      = title_el.text.strip() if title_el is not None and title_el.text else ''
        medline_ta = rec.findtext('MedlineTA','')
        linking    = normalize_issn(rec.findtext('ISSNLinking'))
        print_issn = e_issn = None
        for issn_el in rec.findall('.//ISSN'):
            t   = issn_el.get('IssnType','').lower()
            val = normalize_issn(issn_el.text)
            if t=='print' and val:        print_issn = val
            elif t=='electronic' and val: e_issn     = val
        records.append({'nlm_id':nlm_id,'nlm_title':title,'medline_ta':medline_ta,
                         'nlm_issn':print_issn,'nlm_eissn':e_issn,'nlm_linking':linking})
    return records


def fetch_nlm_journals(query,use_cache=True):
    """Fetch NLM MEDLINE journals via E-utilities or load from timestamped cache."""
    if use_cache and NLM_CACHE.exists():
        with open(NLM_CACHE,'rb') as f:
            cached = pickle.load(f)
        if isinstance(cached,dict):
            df       = cached['data']
            fetch_ts = cached.get('fetched','unknown')
            age_days = (pd.Timestamp.now()-pd.Timestamp(fetch_ts)).days if fetch_ts!='unknown' else None
            print(f"NLM MEDLINE Journals: loaded from cache (fetched {fetch_ts}, {age_days} days ago)")
            if age_days and age_days>90:
                print(f"  ⚠ Cache is {age_days} days old — consider refresh with use_cache=False")
        else:
            df = cached
            print("NLM MEDLINE Journals: loaded from legacy cache (no timestamp) — consider refresh")
        return df

    print(f"Fetching NLM MEDLINE Journals: '{query}'")
    webenv, query_key, count = esearch_nlm(query)
    print(f"  Total records: {count:,}")

    all_records = []
    for start in tqdm(range(0,count,BATCH_SIZE),desc="Fetching NLM MEDLINE batches"):
        retmax = min(BATCH_SIZE, count-start)
        for attempt in range(3):
            try:
                all_records.extend(parse_nlm_xml(efetch_batch(webenv,query_key,start,retmax)))
                break
            except requests.HTTPError as e:
                print(f"  HTTP error start={start} attempt {attempt+1}: {e}")
                time.sleep(2**attempt)
        time.sleep(SLEEP_SEC)

    df = pd.DataFrame(all_records).reset_index(drop=True)
    print(f"  Records parsed: {len(df):,}")
    with open(NLM_CACHE,'wb') as f:
        pickle.dump({'data':df,'fetched':pd.Timestamp.now().isoformat()},f)
    return df


nlm_df = fetch_nlm_journals(NLM_QUERY, use_cache=True)
print(f"NLM MEDLINE Journals loaded: {len(nlm_df):,}")


Fetching NLM MEDLINE Journals: 'currentlyindexed[All]'
  Total records: 5,227


Fetching NLM MEDLINE batches:   0%|          | 0/11 [00:00<?, ?it/s]

  Records parsed: 5,227
NLM MEDLINE Journals loaded: 5,227


## 11. Build ISSN Lookup Sets

In [10]:
# Build flat ISSN sets for O(1) membership testing.
# NLM MEDLINE: three ISSN fields (print, electronic, linking).
# PMC Journals: two (print, electronic).

pmc_journal_issns = set()
for _, row in pmc_df.iterrows():
    for v in [row.get('pmc_issn_norm'), row.get('pmc_eissn_norm')]:
        if pd.notna(v) and v: pmc_journal_issns.add(v)

nlm_medline_issns = set()
for _, row in nlm_df.iterrows():
    for col in ['nlm_issn','nlm_eissn','nlm_linking']:
        v = row.get(col)
        if pd.notna(v) and v: nlm_medline_issns.add(v)

# nih_issns, ta_issns, nu_issns built in earlier sections

print("ISSN set sizes:")
print(f"  NLM MEDLINE Journals : {len(nlm_medline_issns):,}")
print(f"  PMC Journals         : {len(pmc_journal_issns):,}")
print(f"  NIH-Funded Journals  : {len(nih_issns):,}")
print(f"  Northwestern TA      : {len(ta_issns):,}")
print(f"  Northwestern Pubs    : {len(nu_issns):,}")

print("\nPairwise intersections (ISSN-level):")
print(f"  NLM MEDLINE ∩ PMC Journals   : {len(nlm_medline_issns & pmc_journal_issns):,}")
print(f"  NLM MEDLINE ∩ NIH-Funded     : {len(nlm_medline_issns & nih_issns):,}")
print(f"  NLM MEDLINE ∩ NU TA          : {len(nlm_medline_issns & ta_issns):,}")
print(f"  NLM MEDLINE ∩ NU Pubs        : {len(nlm_medline_issns & nu_issns):,}")
print(f"  PMC Journals ∩ NIH-Funded    : {len(pmc_journal_issns & nih_issns):,}")
print(f"  PMC Journals ∩ NU TA         : {len(pmc_journal_issns & ta_issns):,}")
print(f"  PMC Journals ∩ NU Pubs       : {len(pmc_journal_issns & nu_issns):,}")
print(f"  NIH-Funded ∩ NU TA           : {len(nih_issns & ta_issns):,}")
print(f"  NIH-Funded ∩ NU Pubs         : {len(nih_issns & nu_issns):,}")
print(f"  All five sources             : {len(nlm_medline_issns & pmc_journal_issns & nih_issns & ta_issns & nu_issns):,}")


ISSN set sizes:
  NLM MEDLINE Journals : 9,726
  PMC Journals         : 6,756
  NIH-Funded Journals  : 3,860
  Northwestern TA      : 3,888
  Northwestern Pubs    : 2,522

Pairwise intersections (ISSN-level):
  NLM MEDLINE ∩ PMC Journals   : 2,415
  NLM MEDLINE ∩ NIH-Funded     : 2,967
  NLM MEDLINE ∩ NU TA          : 1,097
  NLM MEDLINE ∩ NU Pubs        : 1,833
  PMC Journals ∩ NIH-Funded    : 1,331
  PMC Journals ∩ NU TA         : 375
  PMC Journals ∩ NU Pubs       : 934
  NIH-Funded ∩ NU TA           : 452
  NIH-Funded ∩ NU Pubs         : 1,558
  All five sources             : 57


## 12. Classify Journals

Each journal row receives a True/False flag for membership in each source.

**Column naming conventions:**
- `in_pmc_journals` = True if the journal appears on the PMC Journals list (has a formal PMC deposit agreement)
- `in_nlm_medline` = True if the journal is currently indexed in MEDLINE
- `in_northwestern_ta` = True if the journal is covered by a Northwestern TA (APC waived)
- `in_northwestern_pubs` = True if NU researchers have published NIH-funded papers in this journal
- `has_pmc_agreement` = unified flag used in APC classification (True for all PMC Journals rows, and for NLM MEDLINE rows that also appear in PMC Journals)


In [ ]:
NLM_ISSN_COLS = ['nlm_issn','nlm_eissn','nlm_linking']
PMC_ISSN_COLS = ['pmc_issn_norm','pmc_eissn_norm']
NIH_ISSN_COLS = ['nih_issn1','nih_issn2','nih_issn3','nih_issn4']

# ── NLM MEDLINE Journals classification ──────────────────────────────────────
nlm_df['in_pmc_journals']       = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, pmc_journal_issns), axis=1)
nlm_df['in_nih_funded']         = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, nih_issns),         axis=1)
nlm_df['in_northwestern_ta']    = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, ta_issns),          axis=1)
nlm_df['in_northwestern_pubs']  = nlm_df.apply(lambda r: any_issn_in_set(r, NLM_ISSN_COLS, nu_issns),          axis=1)
# For APC classification: does this journal have a PMC deposit agreement?
nlm_df['has_pmc_agreement']     = nlm_df['in_pmc_journals']

print("NLM MEDLINE Journals classification:")
print(f"  in PMC Journals       : {nlm_df['in_pmc_journals'].sum():,}")
print(f"  in NIH-Funded list    : {nlm_df['in_nih_funded'].sum():,}")
print(f"  in Northwestern TA    : {nlm_df['in_northwestern_ta'].sum():,}")
print(f"  in Northwestern Pubs  : {nlm_df['in_northwestern_pubs'].sum():,}")
print(f"  in NIH-Funded AND TA  : {(nlm_df['in_nih_funded'] & nlm_df['in_northwestern_ta']).sum():,}")
print(f"  in NU Pubs AND TA     : {(nlm_df['in_northwestern_pubs'] & nlm_df['in_northwestern_ta']).sum():,}")

# ── PMC Journals classification ───────────────────────────────────────────────
pmc_df['in_nlm_medline']        = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, nlm_medline_issns), axis=1)
pmc_df['in_nih_funded']         = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, nih_issns),         axis=1)
pmc_df['in_northwestern_ta']    = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, ta_issns),          axis=1)
pmc_df['in_northwestern_pubs']  = pmc_df.apply(lambda r: any_issn_in_set(r, PMC_ISSN_COLS, nu_issns),          axis=1)
# All rows in pmc_df ARE PMC Journals by definition
pmc_df['has_pmc_agreement']     = True

print("\nPMC Journals classification:")
print(f"  in NLM MEDLINE        : {pmc_df['in_nlm_medline'].sum():,}")
print(f"  in NIH-Funded list    : {pmc_df['in_nih_funded'].sum():,}")
print(f"  in Northwestern TA    : {pmc_df['in_northwestern_ta'].sum():,}")
print(f"  in Northwestern Pubs  : {pmc_df['in_northwestern_pubs'].sum():,}")

# ── NIH-Funded classification ─────────────────────────────────────────────────
nih_df['in_nlm_medline']        = nih_df.apply(lambda r: any_issn_in_set(r, NIH_ISSN_COLS, nlm_medline_issns), axis=1)
nih_df['in_pmc_journals']       = nih_df.apply(lambda r: any_issn_in_set(r, NIH_ISSN_COLS, pmc_journal_issns), axis=1)
nih_df['in_northwestern_ta']    = nih_df.apply(lambda r: any_issn_in_set(r, NIH_ISSN_COLS, ta_issns),          axis=1)
nih_df['in_northwestern_pubs']  = nih_df.apply(lambda r: any_issn_in_set(r, NIH_ISSN_COLS, nu_issns),          axis=1)


## 13. Join Metadata

For each journal row, join metadata from all other sources by ISSN match.  
Journals with no match receive `None` for all columns from the unmatched source.


In [ ]:
def get_nih_meta(row, issn_cols):
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in nih_lookup:
            return nih_lookup[v]
    return NIH_EMPTY.copy()

def get_ta_meta(row, issn_cols):
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in ta_lookup:
            return ta_lookup[v]
    return TA_EMPTY.copy()

def get_nu_meta(row, issn_cols):
    for col in issn_cols:
        v = row.get(col)
        if pd.notna(v) and v and v in nu_lookup:
            return nu_lookup[v]
    return NU_EMPTY.copy()


# ── Join to NLM MEDLINE frame ─────────────────────────────────────────────────
nlm_df = pd.concat([
    nlm_df.reset_index(drop=True),
    pd.DataFrame(nlm_df.apply(lambda r: get_nih_meta(r, NLM_ISSN_COLS), axis=1).tolist()),
    pd.DataFrame(nlm_df.apply(lambda r: get_ta_meta(r,  NLM_ISSN_COLS), axis=1).tolist()),
    pd.DataFrame(nlm_df.apply(lambda r: get_nu_meta(r,  NLM_ISSN_COLS), axis=1).tolist()),
], axis=1)

# ── Join to PMC Journals frame ────────────────────────────────────────────────
# All three metadata joins use pmc_df and PMC_ISSN_COLS. Do NOT substitute
# nlm_df here — doing so would misalign rows (different row counts) and assign
# NLM-row metadata to PMC rows.
pmc_df = pd.concat([
    pmc_df.reset_index(drop=True),
    pd.DataFrame(pmc_df.apply(lambda r: get_nih_meta(r, PMC_ISSN_COLS), axis=1).tolist()),
    pd.DataFrame(pmc_df.apply(lambda r: get_ta_meta(r,  PMC_ISSN_COLS), axis=1).tolist()),
    pd.DataFrame(pmc_df.apply(lambda r: get_nu_meta(r,  PMC_ISSN_COLS), axis=1).tolist()),
], axis=1)

# Post-join integrity assertions
assert len(pmc_df) < len(nlm_df) + 1000, (
    f"pmc_df has {len(pmc_df)} rows — larger than expected. Check for join row-count mismatch.")

print(f"NLM MEDLINE Journals frame shape after joins: {nlm_df.shape}")
print(f"PMC Journals frame shape after joins        : {pmc_df.shape}")

nu_nlm_populated = nlm_df['nu_pub_count'].notna().sum()
nu_pmc_populated = pmc_df['nu_pub_count'].notna().sum()
print(f"\nNU pub_count populated — NLM MEDLINE: {nu_nlm_populated:,}  PMC Journals: {nu_pmc_populated:,}")
print(f"  (These should match in_northwestern_pubs counts from Section 12)")


## 14. APC Liability Classification

Each journal is assigned one of the following liability categories. This determines whether APC exposure is calculated.

### Classification logic

| OA Status | In TA? | Has PMC agreement? | Embargo? | Category | Rationale |
|---|---|---|---|---|---|
| any | Yes | — | — | TA waived ($0) | APC waived under Northwestern TA |
| diamond / S2O | No | — | — | No APC (diamond/S2O) | No APC business model |
| gold | No | — | — | APC required (gold OA) | Gold journals always charge APC |
| hybrid | No | Yes | immediate | Likely $0 (hybrid, PMC immediate) | PMC deposit satisfies NIH public access without paid APC |
| hybrid | No | Yes | embargo > 0 | Likely APC required (hybrid, PMC embargo) | PMC deposit delayed — compliance likely requires APC |
| hybrid | No | No | — | Likely APC required (hybrid, no PMC agreement) | No PMC route — compliance requires APC |
| unknown/other | No | — | — | Unknown | OA status not in NIH top 2,228 list |

### Key embedded assumptions
1. **TA waiver overrides all other conditions.**
2. **OA status comes from the NIH file** (Dimensions, Jul 2025). Journals not in the NIH top 2,228 have no OA status in this dataset.
3. **"PMC agreement with immediate release"** is treated as satisfying NIH public access. This assumes the author relies on journal deposit rather than paying for hybrid OA. Authors who voluntarily pay hybrid OA in these journals are not captured.
4. **`has_pmc_agreement`** is the unified flag used here — True for all PMC Journals rows, and for NLM MEDLINE rows that also appear in PMC Journals.


In [ ]:
def classify_apc_liability(row):
    """
    Assign APC liability category. See logic table in markdown cell above.
    Uses `has_pmc_agreement` to handle both NLM MEDLINE and PMC-only rows correctly.
    """
    oa      = str(row.get('nih_oa_status') or '').lower().strip()
    in_ta   = bool(row.get('in_northwestern_ta', False))
    in_pmc  = bool(row.get('has_pmc_agreement', False))
    embargo = str(row.get('pmc_embargo') or '').lower().strip()

    if in_ta:
        return 'TA waived ($0)'
    if oa in ('diamond', 's2o', 'subscribe to open'):
        return 'No APC (diamond/S2O)'
    if oa == 'gold':
        return 'APC required (gold OA)'
    if oa == 'hybrid':
        if in_pmc:
            if 'immediate' in embargo or embargo == '0 months':
                return 'Likely $0 (hybrid, PMC immediate release)'
            else:
                return 'Likely APC required (hybrid, PMC embargo)'
        else:
            return 'Likely APC required (hybrid, no PMC agreement)'
    if not oa or oa in ('nan', ''):
        return 'Unknown (no OA status — not in NIH top 2,228)'
    return f'Unknown ({oa})'


def compute_financial_cols(df, nu_years=1):
    """
    Add APC liability classification and financial estimate columns.

    Annual average estimate (PRIMARY — use for planning):
        est_annual_apc_[cap] = (nu_pub_count / nu_years) × apc_[cap]
        Interpretation: At 2025 list prices, estimated annual APC exposure
        if NU publishes at the same average rate going forward.

    Multi-year total estimate (SECONDARY — reference/context only):
        total_apc_all_years_[cap] = nu_pub_count × apc_[cap]
        Interpretation: If every paper in the dataset (FY2020–present)
        had been published in 2025 at 2025 list prices. NOT actual expenditure.

    Both are NaN where:
      - Journal is classified as no-charge (TA waived, diamond/S2O, PMC immediate)
      - APC price or pub count is missing

    nu_years: number of distinct publication years in NU dataset (for annualization).
    """
    df = df.copy()
    df['apc_liability'] = df.apply(classify_apc_liability, axis=1)

    apc     = pd.to_numeric(df.get('nih_apc_2025_usd',  pd.Series(dtype=float)), errors='coerce')
    pub_cnt = pd.to_numeric(df.get('nu_pub_count',      pd.Series(dtype=float)), errors='coerce')

    # fillna('') ensures NaN liability values are treated as charge-applicable (conservative)
    no_charge = df['apc_liability'].fillna('').str.startswith(
        ('TA waived', 'No APC', 'Likely $0')
    )

    ann_cnt = pub_cnt / nu_years  # average annual publications per journal

    # ── Annual average (PRIMARY) ──────────────────────────────────────────────
    df['est_annual_apc_2025_price']   = (apc                .clip() * ann_cnt).where(~no_charge)
    df['est_annual_apc_2k_cap']       = (apc.clip(upper=2000) * ann_cnt).where(~no_charge)
    df['est_annual_apc_3k_cap']       = (apc.clip(upper=3000) * ann_cnt).where(~no_charge)
    df['est_annual_apc_6k_cap']       = (apc.clip(upper=6000) * ann_cnt).where(~no_charge)

    # ── Multi-year total (SECONDARY) ──────────────────────────────────────────
    df['total_apc_all_years_2025_price'] = (apc                .clip() * pub_cnt).where(~no_charge)
    df['total_apc_all_years_2k_cap']     = (apc.clip(upper=2000) * pub_cnt).where(~no_charge)
    df['total_apc_all_years_3k_cap']     = (apc.clip(upper=3000) * pub_cnt).where(~no_charge)
    df['total_apc_all_years_6k_cap']     = (apc.clip(upper=6000) * pub_cnt).where(~no_charge)

    return df


nlm_df = compute_financial_cols(nlm_df, nu_years=nu_years)
pmc_df = compute_financial_cols(pmc_df, nu_years=nu_years)

print("APC liability — NLM MEDLINE (NU-published journals only):")
print(nlm_df.loc[nlm_df['in_northwestern_pubs'], 'apc_liability'].value_counts().to_string())
print("\nAPC liability — PMC Journals only (not in NLM MEDLINE, NU-published):")
mask_pmc_only_nu = pmc_df['in_northwestern_pubs'] & ~pmc_df['in_nlm_medline']
print(pmc_df.loc[mask_pmc_only_nu, 'apc_liability'].value_counts().to_string())


## 15. Validation Checks

In [ ]:
print("=" * 68)
print("VALIDATION REPORT")
print("=" * 68)
issues = []

print("\n[1] Row count plausibility")
for label, n, lo, hi in [
    ("NLM MEDLINE Journals", len(nlm_df), 4000, 8000),
    ("PMC Journals",         len(pmc_df), 3000, 8000),
    ("NIH-Funded Journals",  len(nih_df), 2000, 2500),
    ("NU unique journals",   len(nu_agg), 100,  5000),
]:
    ok = lo <= n <= hi
    print(f"  [{'OK' if ok else 'CHECK'}] {label}: {n:,}" +
          (f" (expected {lo:,}–{hi:,})" if not ok else ""))
    if not ok: issues.append(f"Row count outside range: {label}={n:,}")

print("\n[2] PMC Journals frame row-count integrity (guard against join bug)")
if len(pmc_df) >= len(nlm_df):
    msg = f"pmc_df ({len(pmc_df):,}) >= nlm_df ({len(nlm_df):,}) — possible join row-count mismatch"
    print(f"  [WARNING] {msg}")
    issues.append(msg)
else:
    print(f"  [OK] pmc_df ({len(pmc_df):,}) < nlm_df ({len(nlm_df):,})")

print("\n[3] NU metadata join integrity")
for fname, df_check, flag_col in [
    ("NLM MEDLINE", nlm_df, 'in_northwestern_pubs'),
    ("PMC Journals", pmc_df, 'in_northwestern_pubs'),
]:
    in_pubs = df_check[flag_col].sum()
    pop     = df_check['nu_pub_count'].notna().sum()
    ok = in_pubs == pop
    print(f"  [{'OK' if ok else 'CHECK'}] {fname}: in_northwestern_pubs={in_pubs:,}, nu_pub_count populated={pop:,}")
    if not ok: issues.append(f"{fname} NU join mismatch: {in_pubs} vs {pop}")

print("\n[4] NU publication APC data coverage (financial blind spot quantification)")
total_nu = nu_agg['nu_pub_count'].sum()
covered_nlm = nlm_df.loc[nlm_df['in_nih_funded'], 'nu_pub_count'].sum()
covered_pmc = pmc_df.loc[pmc_df['in_nih_funded'] & ~pmc_df['in_nlm_medline'], 'nu_pub_count'].sum()
total_covered = covered_nlm + covered_pmc
pct = total_covered / total_nu * 100 if total_nu > 0 else 0
print(f"  Total NU-NIH publications (all years): {total_nu:,.0f}")
print(f"  In journals with APC data             : {total_covered:,.0f} ({pct:.1f}%)")
print(f"  WITHOUT APC data (blind spot)         : {total_nu - total_covered:,.0f} ({100-pct:.1f}%)")
if pct < 70:
    issues.append(f"Only {pct:.0f}% of NU pubs have APC data — financial estimates are substantially incomplete")
    print(f"  ⚠ WARNING: coverage below 70% — totals significantly understate exposure")
else:
    print(f"  Coverage sufficient for order-of-magnitude estimates.")

print("\n[5] No negative financial values")
fin_cols = [c for c in nlm_df.columns if ('est_annual_apc' in c or 'total_apc' in c)]
neg_found = any((pd.to_numeric(nlm_df[c],errors='coerce')<0).sum()>0 for c in fin_cols)
print(f"  [{'FAIL' if neg_found else 'OK'}] {'Negative values found — check financial columns' if neg_found else 'No negative values in financial columns'}")
if neg_found: issues.append("Negative financial values detected")

print("\n[6] APC classification spot-check (Nature Communications eISSN 2041-1723)")
nc = nlm_df[nlm_df['nlm_eissn']=='2041-1723']
if len(nc)==1:
    r = nc.iloc[0]
    for f in ['has_pmc_agreement','in_nih_funded','in_northwestern_ta','nih_oa_status','nih_apc_2025_usd','apc_liability']:
        print(f"  {f}: {r.get(f)}")
elif len(nc)==0:
    print("  Nature Communications not in NLM MEDLINE frame (check PMC Journals frame)")
else:
    print(f"  ⚠ {len(nc)} rows matched — expected 1")

print("\n[7] per-year data availability")
if len(nu_yearly_agg) > 0:
    yr_counts = nu_yearly_agg.groupby('pub_year')['nu_pub_count_year'].sum().sort_index()
    print("  Publications per year in NU dataset:")
    for yr, cnt in yr_counts.items():
        print(f"    {int(yr)}: {int(cnt):,}")
else:
    print("  ⚠ No yearly data available (year column not found)")
    issues.append("Yearly breakdown unavailable — year column not detected in NU file")

print("\n" + "=" * 68)
if issues:
    print(f"VALIDATION COMPLETE — {len(issues)} issue(s) require attention:")
    for i, iss in enumerate(issues,1): print(f"  {i}. {iss}")
else:
    print("VALIDATION COMPLETE — No issues detected.")
print("=" * 68)


## 16. Build Comparison Tables

Journals are sliced into mutually exclusive groups based on their membership flags.  
**NLM MEDLINE–anchored slices** cover journals indexed in MEDLINE.  
**PMC Journals–only slices** cover journals in the PMC Journals list but not in NLM MEDLINE.  
**NIH-Funded only** covers journals in the NIH top list not found in either source above.


In [ ]:
APC_FIN_COLS = [
    'apc_liability',
    'est_annual_apc_2025_price','est_annual_apc_2k_cap','est_annual_apc_3k_cap','est_annual_apc_6k_cap',
    'total_apc_all_years_2025_price','total_apc_all_years_2k_cap','total_apc_all_years_3k_cap','total_apc_all_years_6k_cap',
]

NLM_BASE = [
    'nlm_id','nlm_title','medline_ta','nlm_issn','nlm_eissn','nlm_linking',
    'in_pmc_journals','in_nih_funded','in_northwestern_ta','in_northwestern_pubs',
]
PMC_BASE = ['pmc_title','pmc_issn_norm','pmc_eissn_norm',
            'in_nlm_medline','in_nih_funded','in_northwestern_ta','in_northwestern_pubs']
for opt in ['pmc_participation','pmc_nlm_id','pmc_embargo']:
    if opt in pmc_df.columns: PMC_BASE.append(opt)

ALL_META = (
    [c for c in NIH_META_COLS if c in nlm_df.columns] +
    [c for c in TA_META_COLS  if c in nlm_df.columns] +
    [c for c in NU_META_COLS  if c in nlm_df.columns] +
    [c for c in APC_FIN_COLS  if c in nlm_df.columns]
)
seen=set(); ALL_META=[c for c in ALL_META if not (c in seen or seen.add(c))]


def make_nlm_table(mask, label):
    cols = [c for c in NLM_BASE + ALL_META if c in nlm_df.columns]
    df = nlm_df.loc[mask, cols].copy().reset_index(drop=True)
    df.insert(0, 'category', label)
    return df


def make_pmc_table(mask, label):
    pmc_meta = [c for c in ALL_META if c in pmc_df.columns]
    cols = [c for c in PMC_BASE + pmc_meta if c in pmc_df.columns]
    df = pmc_df.loc[mask, cols].copy().reset_index(drop=True)
    df.insert(0, 'category', label)
    return df


# ── NLM MEDLINE–anchored slices ───────────────────────────────────────────────
df_nlm_pmc_nih = make_nlm_table(nlm_df['in_pmc_journals'] & nlm_df['in_nih_funded'],    'NLM MEDLINE + PMC Journals + NIH-Funded')
df_nlm_pmc     = make_nlm_table(nlm_df['in_pmc_journals'] & ~nlm_df['in_nih_funded'],   'NLM MEDLINE + PMC Journals (not NIH-Funded)')
df_nlm_nih     = make_nlm_table(~nlm_df['in_pmc_journals'] & nlm_df['in_nih_funded'],   'NLM MEDLINE + NIH-Funded (not PMC Journals)')
df_nlm_only    = make_nlm_table(~nlm_df['in_pmc_journals'] & ~nlm_df['in_nih_funded'],  'NLM MEDLINE Only')

# ── PMC Journals–only slices ──────────────────────────────────────────────────
df_pmc_nih     = make_pmc_table(~pmc_df['in_nlm_medline'] & pmc_df['in_nih_funded'],    'PMC Journals + NIH-Funded (not NLM MEDLINE)')
df_pmc_only    = make_pmc_table(~pmc_df['in_nlm_medline'] & ~pmc_df['in_nih_funded'],   'PMC Journals Only (not NLM MEDLINE or NIH-Funded)')

# ── NIH-Funded only ───────────────────────────────────────────────────────────
df_nih_only    = nih_df[~nih_df['in_nlm_medline'] & ~nih_df['in_pmc_journals']].copy().reset_index(drop=True)
df_nih_only.insert(0, 'category', 'NIH-Funded Only (not in NLM MEDLINE or PMC Journals)')

# ── Cross-cutting TA view ─────────────────────────────────────────────────────
df_ta_nlm = make_nlm_table(nlm_df['in_northwestern_ta'],                                  'NU TA-Covered — in NLM MEDLINE')
df_ta_pmc = make_pmc_table(pmc_df['in_northwestern_ta'] & ~pmc_df['in_nlm_medline'],       'NU TA-Covered — PMC Journals only')

# ── Cross-cutting NU publications view ───────────────────────────────────────
df_nu_nlm = make_nlm_table(nlm_df['in_northwestern_pubs'],                                'NU-Published — in NLM MEDLINE')
df_nu_pmc = make_pmc_table(pmc_df['in_northwestern_pubs'] & ~pmc_df['in_nlm_medline'],     'NU-Published — PMC Journals only')

# ── Verify NLM slices sum ─────────────────────────────────────────────────────
nlm_slice_sum = len(df_nlm_pmc_nih)+len(df_nlm_pmc)+len(df_nlm_nih)+len(df_nlm_only)
status = "OK" if nlm_slice_sum == len(nlm_df) else f"CHECK — expected {len(nlm_df):,}"
print(f"NLM MEDLINE slice sum: {nlm_slice_sum:,} [{status}]")

print("\nComparison table row counts:")
for label, tbl in [
    ('NLM MEDLINE + PMC Journals + NIH-Funded',              df_nlm_pmc_nih),
    ('NLM MEDLINE + PMC Journals (not NIH-Funded)',          df_nlm_pmc),
    ('NLM MEDLINE + NIH-Funded (not PMC Journals)',          df_nlm_nih),
    ('NLM MEDLINE Only',                                     df_nlm_only),
    ('PMC Journals + NIH-Funded (not NLM MEDLINE)',          df_pmc_nih),
    ('PMC Journals Only',                                    df_pmc_only),
    ('NIH-Funded Only',                                      df_nih_only),
    ('NU TA-Covered — NLM MEDLINE',                          df_ta_nlm),
    ('NU TA-Covered — PMC Journals only',                    df_ta_pmc),
    ('NU-Published — NLM MEDLINE',                           df_nu_nlm),
    ('NU-Published — PMC Journals only',                     df_nu_pmc),
]:
    print(f"  {label:<55}: {len(tbl):>5,}")


## 17. Summary Statistics

In [ ]:
n_nlm = len(nlm_df); n_pmc = len(pmc_df); n_nih = len(nih_df)
pct = lambda n,d: f"{n/d*100:.1f}%" if d>0 else "—"

# Total unique journals where NU has published (NLM + PMC-only)
nu_in_nlm_count      = int(nlm_df['in_northwestern_pubs'].sum())
nu_in_pmc_only_count = int((pmc_df['in_northwestern_pubs'] & ~pmc_df['in_nlm_medline']).sum())
nu_total_unique      = nu_in_nlm_count + nu_in_pmc_only_count

summary_rows = [
    {'Category':'In NLM MEDLINE + PMC Journals + NIH-Funded',
     'Count':len(df_nlm_pmc_nih),
     '% of NLM MEDLINE':pct(len(df_nlm_pmc_nih),n_nlm),
     '% of PMC Journals':pct(len(df_nlm_pmc_nih),n_pmc),
     '% of NIH-Funded':pct(len(df_nlm_pmc_nih),n_nih)},
    {'Category':'In NLM MEDLINE + PMC Journals (not NIH-Funded)',
     'Count':len(df_nlm_pmc),
     '% of NLM MEDLINE':pct(len(df_nlm_pmc),n_nlm),
     '% of PMC Journals':pct(len(df_nlm_pmc),n_pmc),
     '% of NIH-Funded':'—'},
    {'Category':'In NLM MEDLINE + NIH-Funded (not PMC Journals)',
     'Count':len(df_nlm_nih),
     '% of NLM MEDLINE':pct(len(df_nlm_nih),n_nlm),
     '% of PMC Journals':'—',
     '% of NIH-Funded':pct(len(df_nlm_nih),n_nih)},
    {'Category':'In NLM MEDLINE Only',
     'Count':len(df_nlm_only),
     '% of NLM MEDLINE':pct(len(df_nlm_only),n_nlm),
     '% of PMC Journals':'—','% of NIH-Funded':'—'},
    {'Category':'In PMC Journals + NIH-Funded (not NLM MEDLINE)',
     'Count':len(df_pmc_nih),
     '% of NLM MEDLINE':'—',
     '% of PMC Journals':pct(len(df_pmc_nih),n_pmc),
     '% of NIH-Funded':pct(len(df_pmc_nih),n_nih)},
    {'Category':'In PMC Journals Only (not NLM MEDLINE or NIH-Funded)',
     'Count':len(df_pmc_only),
     '% of NLM MEDLINE':'—',
     '% of PMC Journals':pct(len(df_pmc_only),n_pmc),
     '% of NIH-Funded':'—'},
    {'Category':'In NIH-Funded Only',
     'Count':len(df_nih_only),
     '% of NLM MEDLINE':'—','% of PMC Journals':'—',
     '% of NIH-Funded':pct(len(df_nih_only),n_nih)},
    {'Category':'─'*50,'Count':'','% of NLM MEDLINE':'','% of PMC Journals':'','% of NIH-Funded':''},
    {'Category':'NU TA-covered — in NLM MEDLINE',
     'Count':int(nlm_df['in_northwestern_ta'].sum()),
     '% of NLM MEDLINE':pct(int(nlm_df['in_northwestern_ta'].sum()),n_nlm),
     '% of PMC Journals':'—','% of NIH-Funded':'—'},
    {'Category':'NU TA-covered — in PMC Journals',
     'Count':int(pmc_df['in_northwestern_ta'].sum()),
     '% of NLM MEDLINE':'—',
     '% of PMC Journals':pct(int(pmc_df['in_northwestern_ta'].sum()),n_pmc),
     '% of NIH-Funded':'—'},
    {'Category':'NU TA-covered — in NIH-Funded list',
     'Count':int(nih_df['in_northwestern_ta'].sum()),
     '% of NLM MEDLINE':'—','% of PMC Journals':'—',
     '% of NIH-Funded':pct(int(nih_df['in_northwestern_ta'].sum()),n_nih)},
    {'Category':'─'*50,'Count':'','% of NLM MEDLINE':'','% of PMC Journals':'','% of NIH-Funded':''},
    {'Category':'NU publications — in NLM MEDLINE Journals',
     'Count':nu_in_nlm_count,
     '% of NLM MEDLINE':pct(nu_in_nlm_count,n_nlm),
     '% of PMC Journals':'—','% of NIH-Funded':'—'},
    {'Category':'NU publications — in PMC Journals (not NLM MEDLINE)',
     'Count':nu_in_pmc_only_count,
     '% of NLM MEDLINE':'—',
     '% of PMC Journals':pct(nu_in_pmc_only_count,n_pmc),
     '% of NIH-Funded':'—'},
    {'Category':'NU publications — total unique journals (NLM MEDLINE + PMC only)',
     'Count':nu_total_unique,
     '% of NLM MEDLINE':'—','% of PMC Journals':'—','% of NIH-Funded':'—'},
    {'Category':'─'*50,'Count':'','% of NLM MEDLINE':'','% of PMC Journals':'','% of NIH-Funded':''},
    {'Category':'Total NLM MEDLINE Journals',
     'Count':n_nlm,'% of NLM MEDLINE':'100%','% of PMC Journals':'—','% of NIH-Funded':'—'},
    {'Category':'Total PMC Journals',
     'Count':n_pmc,'% of NLM MEDLINE':'—','% of PMC Journals':'100%','% of NIH-Funded':'—'},
    {'Category':'Total NIH-Funded Journals (top 2,228)',
     'Count':n_nih,'% of NLM MEDLINE':'—','% of PMC Journals':'—','% of NIH-Funded':'100%'},
    {'Category':'Total Northwestern TA journals (unique ISSNs)',
     'Count':len(ta_issns),'% of NLM MEDLINE':'—','% of PMC Journals':'—','% of NIH-Funded':'—'},
]
summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
summary


## 18. Financial Summary and Per-Year APC Analysis

### APC estimate columns — what they mean

| Column prefix | Time basis | Formula | Use for |
|---|---|---|---|
| `est_annual_apc_*` | **Annual average** | `(nu_pub_count ÷ nu_years) × apc_price` | Planning / budgeting |
| `total_apc_all_years_*` | **Multi-year total** | `nu_pub_count × apc_price` | Context / magnitude |

**Important:** Both use **2025 list prices** regardless of actual publication year. Historical prices are not available.

### Per-year APC breakdown

The per-year table applies 2025 APC prices to each year's actual publication counts. This allows you to:
- See whether NU's publication volume in these journals is growing or declining
- Identify peak-exposure years
- Understand what the trend implies for future exposure

**Critical caveat for per-year table:** Using 2025 prices on 2020 papers overstates historical costs (APCs have generally risen). Using 2025 prices on future years is a planning assumption. The table should be read as *"if these papers had been published in 2025 at 2025 list prices, what would each year's exposure have been?"*


In [ ]:
def _build_financial_summary(nlm_df, pmc_df):
    """
    Aggregate APC exposure by liability category for NU-published journals.
    Includes both annual average and multi-year total.
    PMC-only rows are included to avoid double-counting NLM-matched rows.
    """
    nlm_nu = nlm_df[nlm_df['in_northwestern_pubs']].copy()
    pmc_nu = pmc_df[pmc_df['in_northwestern_pubs'] & ~pmc_df['in_nlm_medline']].copy()

    rows = []
    for source_label, frame in [('NLM MEDLINE–matched', nlm_nu), ('PMC Journals–only', pmc_nu)]:
        for cat, grp in frame.groupby('apc_liability', dropna=False):
            row = {'source': source_label, 'apc_liability': cat, 'journal_count': len(grp)}
            for c in ['nu_pub_count',
                      'est_annual_apc_2025_price','est_annual_apc_2k_cap',
                      'est_annual_apc_3k_cap','est_annual_apc_6k_cap',
                      'total_apc_all_years_2025_price','total_apc_all_years_2k_cap',
                      'total_apc_all_years_3k_cap','total_apc_all_years_6k_cap']:
                row[c] = pd.to_numeric(grp.get(c, pd.Series()), errors='coerce').sum() if c in grp.columns else None
            rows.append(row)

    df_out = pd.DataFrame(rows).sort_values(['source','apc_liability'])

    # Add grand total row
    num_cols = [c for c in df_out.columns if c not in ('source','apc_liability')]
    totals = df_out[num_cols].sum(numeric_only=True).to_dict()
    totals.update({'source':'TOTAL — ALL CATEGORIES', 'apc_liability':'—'})
    df_out = pd.concat([df_out, pd.DataFrame([totals])], ignore_index=True)

    return df_out


def _build_per_year_apc(nu_yearly_agg, nlm_df, pmc_df):
    """
    For each publication year, calculate estimated APC exposure at 2025 prices.

    Method:
      1. For each (issn, year) record in nu_yearly_agg, look up the 2025 APC price
         and liability category from nlm_df (primary) or pmc_df (fallback).
      2. Compute yearly exposure = pub_count_year × apc_price (if charge-applicable).
      3. Aggregate by year.

    All prices are 2025 list prices applied uniformly across all years.
    This does NOT reflect actual historical prices paid.
    """
    if len(nu_yearly_agg) == 0:
        return pd.DataFrame(columns=['pub_year','nu_pub_count_year',
                                     'journals_with_apc_data',
                                     'est_apc_2025_price','est_apc_2k_cap',
                                     'est_apc_3k_cap','est_apc_6k_cap'])

    # Build a per-ISSN price + liability lookup from NLM (primary) then PMC (fallback)
    price_lookup = {}   # issn -> {apc_2025_usd, apc_2k, apc_3k, apc_6k, liability}

    def add_to_lookup(df_source, issn_cols):
        for _, row in df_source.iterrows():
            for c in issn_cols:
                v = row.get(c)
                if pd.notna(v) and v and v not in price_lookup:
                    apc = pd.to_numeric(row.get('nih_apc_2025_usd'), errors='coerce')
                    lib = row.get('apc_liability', '')
                    no_chg = str(lib).startswith(('TA waived','No APC','Likely $0'))
                    price_lookup[v] = {
                        'apc'        : np.nan if (pd.isna(apc) or no_chg) else float(apc),
                        'apc_2k'     : np.nan if (pd.isna(apc) or no_chg) else min(float(apc),2000),
                        'apc_3k'     : np.nan if (pd.isna(apc) or no_chg) else min(float(apc),3000),
                        'apc_6k'     : np.nan if (pd.isna(apc) or no_chg) else min(float(apc),6000),
                        'liability'  : lib,
                    }

    add_to_lookup(nlm_df, NLM_ISSN_COLS)
    add_to_lookup(pmc_df, PMC_ISSN_COLS)

    # Merge price data onto yearly aggregate
    yr = nu_yearly_agg.copy()
    yr['apc']     = yr['issn_norm'].map(lambda x: price_lookup.get(x,{}).get('apc'))
    yr['apc_2k']  = yr['issn_norm'].map(lambda x: price_lookup.get(x,{}).get('apc_2k'))
    yr['apc_3k']  = yr['issn_norm'].map(lambda x: price_lookup.get(x,{}).get('apc_3k'))
    yr['apc_6k']  = yr['issn_norm'].map(lambda x: price_lookup.get(x,{}).get('apc_6k'))

    yr['est_apc_2025_price'] = yr['apc']    * yr['nu_pub_count_year']
    yr['est_apc_2k_cap']     = yr['apc_2k'] * yr['nu_pub_count_year']
    yr['est_apc_3k_cap']     = yr['apc_3k'] * yr['nu_pub_count_year']
    yr['est_apc_6k_cap']     = yr['apc_6k'] * yr['nu_pub_count_year']

    yr_summary = yr.groupby('pub_year').agg(
        nu_pub_count_year     = ('nu_pub_count_year',   'sum'),
        journals_with_apc_data= ('apc',                  lambda x: x.notna().sum()),
        est_apc_2025_price    = ('est_apc_2025_price',   'sum'),
        est_apc_2k_cap        = ('est_apc_2k_cap',       'sum'),
        est_apc_3k_cap        = ('est_apc_3k_cap',       'sum'),
        est_apc_6k_cap        = ('est_apc_6k_cap',       'sum'),
    ).reset_index().rename(columns={'pub_year':'pub_year'})

    yr_summary['pub_year'] = yr_summary['pub_year'].astype(int)

    # Add a totals row
    totals = yr_summary[[c for c in yr_summary.columns if c!='pub_year']].sum(numeric_only=True).to_dict()
    totals['pub_year'] = 'TOTAL (all years)'
    yr_summary = pd.concat([yr_summary, pd.DataFrame([totals])], ignore_index=True)

    return yr_summary


def _build_apc_breakdown(nlm_df, pmc_df):
    """Journal-level APC exposure sorted by annual estimate descending."""
    keep_nlm = ['nlm_title','nlm_issn','nlm_eissn',
                'nih_oa_status','nih_publisher','nih_publisher_type','nih_apc_2025_usd','nih_apc_category',
                'in_pmc_journals','pmc_embargo','in_northwestern_ta','northwestern_ta_agreement',
                'nu_pub_count','nu_unique_grants','apc_liability',
                'est_annual_apc_2025_price','est_annual_apc_2k_cap','est_annual_apc_3k_cap','est_annual_apc_6k_cap',
                'total_apc_all_years_2025_price']
    keep_pmc = ['pmc_title','pmc_issn_norm','pmc_eissn_norm',
                'nih_oa_status','nih_publisher','nih_publisher_type','nih_apc_2025_usd','nih_apc_category',
                'pmc_embargo','in_northwestern_ta','northwestern_ta_agreement',
                'nu_pub_count','nu_unique_grants','apc_liability',
                'est_annual_apc_2025_price','est_annual_apc_2k_cap','est_annual_apc_3k_cap','est_annual_apc_6k_cap',
                'total_apc_all_years_2025_price']

    nlm_rows = nlm_df[nlm_df['in_northwestern_pubs']][[c for c in keep_nlm if c in nlm_df.columns]].copy()
    nlm_rows.insert(0,'source','NLM MEDLINE')

    pmc_rows = pmc_df[pmc_df['in_northwestern_pubs'] & ~pmc_df['in_nlm_medline']][[c for c in keep_pmc if c in pmc_df.columns]].copy()
    pmc_rows = pmc_rows.rename(columns={'pmc_title':'nlm_title','pmc_issn_norm':'nlm_issn','pmc_eissn_norm':'nlm_eissn'})
    pmc_rows.insert(0,'source','PMC Journals only')

    bd = pd.concat([nlm_rows, pmc_rows], ignore_index=True)
    bd = bd.sort_values('est_annual_apc_2025_price', ascending=False, na_position='last')
    return bd


def _build_logic_table(nu_years, nu_year_min, nu_year_max):
    logic = pd.DataFrame([
        {'OA Status':'any',         'In TA?':'Yes','Has PMC agreement?':'—','Embargo?':'—',
         'APC Liability Category':'TA waived ($0)',
         'Notes':'Covered by Northwestern Wiley or Springer Nature BTAA agreement'},
        {'OA Status':'diamond/S2O', 'In TA?':'No', 'Has PMC agreement?':'—','Embargo?':'—',
         'APC Liability Category':'No APC (diamond/S2O)',
         'Notes':'Diamond and Subscribe to Open journals never charge APCs'},
        {'OA Status':'gold',        'In TA?':'No', 'Has PMC agreement?':'—','Embargo?':'—',
         'APC Liability Category':'APC required (gold OA)',
         'Notes':'Gold OA always requires APC unless TA-covered'},
        {'OA Status':'hybrid',      'In TA?':'No', 'Has PMC agreement?':'Yes','Embargo?':'immediate',
         'APC Liability Category':'Likely $0 (hybrid, PMC immediate release)',
         'Notes':'Journal has PMC deposit agreement with immediate release — satisfies NIH public access without paid OA APC'},
        {'OA Status':'hybrid',      'In TA?':'No', 'Has PMC agreement?':'Yes','Embargo?':'embargo > 0',
         'APC Liability Category':'Likely APC required (hybrid, PMC embargo)',
         'Notes':'PMC deposit delayed — compliance likely requires APC payment'},
        {'OA Status':'hybrid',      'In TA?':'No', 'Has PMC agreement?':'No', 'Embargo?':'—',
         'APC Liability Category':'Likely APC required (hybrid, no PMC agreement)',
         'Notes':'No PMC deposit route — compliance requires APC'},
        {'OA Status':'unknown',     'In TA?':'No', 'Has PMC agreement?':'—','Embargo?':'—',
         'APC Liability Category':'Unknown (no OA status — not in NIH top 2,228)',
         'Notes':'Journal not in NIH top 2,228 list — OA status unavailable in this dataset'},
    ])
    meta = pd.DataFrame([
        {'Parameter':'NU publication years',                'Value':f"{nu_year_min}–{nu_year_max}"},
        {'Parameter':'Number of years in NU dataset',       'Value':str(nu_years)},
        {'Parameter':'NIH-Funded list coverage period',     'Value':'January–July 2025 (7 months)'},
        {'Parameter':'APC price data year',                 'Value':'2025 list prices'},
        {'Parameter':'Annual average formula',              'Value':'(nu_pub_count ÷ nu_years) × nih_apc_2025_usd'},
        {'Parameter':'Multi-year total formula',            'Value':'nu_pub_count × nih_apc_2025_usd'},
        {'Parameter':'Per-year table formula',              'Value':'nu_pub_count_year × nih_apc_2025_usd (2025 prices applied to all years)'},
        {'Parameter':'Annual average — use for',            'Value':'Planning and budgeting: "if we publish at the same average rate going forward"'},
        {'Parameter':'Multi-year total — use for',          'Value':'Context only: not a real expenditure figure'},
        {'Parameter':'Per-year table — use for',            'Value':'Trend analysis; shows whether volume is growing. NOT historical prices.'},
        {'Parameter':'APC data coverage gap',               'Value':'Only journals in NIH top 2,228 have APC data. Publications in other journals are excluded from all financial estimates.'},
        {'Parameter':'TA file limitation',                  'Value':'TA files contain eISSN only — journals with print-only ISSNs may be missed.'},
        {'Parameter':'PMC Journals clarification',          'Value':'"PMC Journals" = journals with formal PMC deposit agreements. This is NOT a list of articles deposited in PMC.'},
        {'Parameter':'NLM MEDLINE clarification',           'Value':'"NLM MEDLINE Journals" = subset of NLM Catalog currently indexed in MEDLINE (~5,200 journals). NOT the full NLM Catalog.'},
    ])
    return {'logic': logic, 'meta': meta}


fin_summary   = _build_financial_summary(nlm_df, pmc_df)
per_year_apc  = _build_per_year_apc(nu_yearly_agg, nlm_df, pmc_df)
apc_breakdown = _build_apc_breakdown(nlm_df, pmc_df)
logic_table   = _build_logic_table(nu_years, nu_year_min, nu_year_max)

print("=== ANNUAL APC EXPOSURE SUMMARY (2025 list prices) ===")
display_cols = ['source','apc_liability','journal_count','nu_pub_count',
                'est_annual_apc_2025_price','est_annual_apc_2k_cap',
                'total_apc_all_years_2025_price']
display_cols = [c for c in display_cols if c in fin_summary.columns]
print(fin_summary[display_cols].to_string(index=False))

if len(per_year_apc) > 0:
    print("\n=== PER-YEAR APC EXPOSURE (2025 prices applied to all years) ===")
    print(per_year_apc.to_string(index=False))


## 19. Export to Excel

In [ ]:
output_path = OUTPUT_DIR / "Galter_NLM_MEDLINE_PMC_NIH_NU_journal_comparison.xlsx"

def drop_cat(df):
    return df.drop(columns=['category'], errors='ignore')

sheets = [
    ('Summary',                      summary),
    ('NLM_PMC_NIH',                  drop_cat(df_nlm_pmc_nih)),
    ('NLM_PMC_notNIH',               drop_cat(df_nlm_pmc)),
    ('NLM_NIH_notPMC',               drop_cat(df_nlm_nih)),
    ('NLM_MEDLINE_Only',             drop_cat(df_nlm_only)),
    ('PMC_NIH_notNLM',               drop_cat(df_pmc_nih)),
    ('PMC_Journals_Only',            drop_cat(df_pmc_only)),
    ('NIH_Funded_Only',              drop_cat(df_nih_only)),
    ('NU_TA_Covered_NLM',            drop_cat(df_ta_nlm)),
    ('NU_TA_Covered_PMC_only',       drop_cat(df_ta_pmc)),
    ('NU_Published_NLM',             drop_cat(df_nu_nlm)),
    ('NU_Published_PMC_only',        drop_cat(df_nu_pmc)),
    ('APC_Financial_Summary',        fin_summary),
    ('APC_Per_Year_Breakdown',       per_year_apc),
    ('APC_Journal_Breakdown',        apc_breakdown),
    ('APC_Logic_Guide',              logic_table),   # dict — two tables
]

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for sheet_name, df in sheets:
        if isinstance(df, dict):
            df['logic'].to_excel(writer, sheet_name=sheet_name, index=False, startrow=0)
            start_row = len(df['logic']) + 3
            df['meta'].to_excel(writer, sheet_name=sheet_name, index=False, startrow=start_row)
        else:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        for col_cells in ws.columns:
            max_len = max((len(str(cell.value)) if cell.value is not None else 0)
                          for cell in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 55)

print(f"Workbook written: {output_path}")
for sheet_name, df in sheets:
    n = len(df['logic'])+len(df['meta']) if isinstance(df,dict) else len(df)
    print(f"  {sheet_name:<32}: {n:>5,} rows")


## Notes, Caveats, and Reproducibility

### Key terminology reminder
- **NLM MEDLINE Journals** — Journals currently indexed in MEDLINE, retrieved via E-utilities query `currentlyindexed[All]`. This is a curated subset (~5,200 journals) of the broader NLM Catalog. It is NOT the full NLM Catalog.
- **PMC Journals** — Journals with formal deposit agreements with PMC. This is NOT a list of articles deposited in PMC. The `in_pmc_journals` flag indicates whether a journal has such an agreement, which is relevant for APC liability classification.

### Cache management
```python
pmc_df = fetch_pmc_journals(use_cache=False)        # Force fresh download
nlm_df = fetch_nlm_journals(NLM_QUERY, use_cache=False)  # Force fresh fetch
```
Local files (NIH, TA, NU) are read fresh every run. Caches include timestamps and warn when >90 days old.

### Changing the NLM MEDLINE scope
To fetch the broader NLM Catalog journal universe, set `NLM_QUERY = "ncbijournals[All]"` in Configuration and delete `cache/nlm_medline_journals.pkl` before re-running.

### APC estimate columns at a glance

| Column pattern | Time basis | Formula | Intended use |
|---|---|---|---|
| `est_annual_apc_*` | Annual average | `(all-year count ÷ n_years) × 2025 price` | **Planning / budgeting** |
| `total_apc_all_years_*` | Multi-year total | `all-year count × 2025 price` | Context / magnitude |
| Per-year table `est_apc_*` | Per year, 2025 prices | `year_count × 2025 price` | Trend analysis |

### Policy use caveats
> All financial figures in this analysis are **planning estimates at 2025 list prices**. They reflect what APC exposure *would be* at current prices — not what was actually paid. Institutional discounts, waivers, and TA coverage reduce actual costs. The APC data coverage gap (publications in journals outside the NIH top 2,228 list) means all totals are **minimum estimates**. The share of NU publications covered by APC pricing data is quantified in the Validation section and must be disclosed when presenting these figures to leadership.
